# Local Data Preparation

Adapted from the supplied work notebook. Requires the original dataset/train/val/test CSVs and local source media, which are not included in this package. This notebook writes generated CSVs and extracts clips. Run sections deliberately; the review notebook is the read-only entry point. Configure `AV_ASD_VIDEO_DIR` before starting the kernel.

In [ ]:
from pathlib import Path
import os
import pandas as pd

# Detect the repository from its root or notebooks directory.
project = Path.cwd()
if not (project / "dataset/csvs").is_dir():
    project = project.parent
if not (project / "dataset/csvs").is_dir():
    raise RuntimeError("Open this notebook from the repository root or notebooks folder.")
csv_folder = project / "dataset/csvs"

# Configure the original video directory using an environment variable.
video_setting = os.environ.get("AV_ASD_VIDEO_DIR")
if not video_setting:
    raise RuntimeError("Set AV_ASD_VIDEO_DIR to your local original video folder first.")
video_folder = Path(video_setting)
if not video_folder.is_dir():
    raise FileNotFoundError(video_folder)


## CSV Cleaning and Split Checks

In [ ]:
from pathlib import Path
import re
import pandas as pd

folder = csv_folder
names = ["dataset", "train", "val", "test"]
cleaned = {}
problems = []

for name in names:
    path = folder / f"{name}.csv"
    df = pd.read_csv(path, dtype={"Video_ID": "string"})
    original_rows = len(df)

    # Strip whitespace from column names and text values.
    df.columns = df.columns.str.strip()
    if df.columns.duplicated().any():
        raise ValueError(f"{name}: duplicate column names milay")

    for col in df.select_dtypes(include=["object", "string"]).columns:
        df[col] = df[col].map(
            lambda value: value.strip() if isinstance(value, str) else value
        )

    # Remove blank rows and exact duplicate rows.
    df = df.replace(r"^\s*$", pd.NA, regex=True)
    empty_rows = int(df.isna().all(axis=1).sum())
    df = df.dropna(how="all")
    duplicate_rows = int(df.duplicated().sum())
    df = df.drop_duplicates().reset_index(drop=True)

    if "Video_ID" not in df.columns:
        problems.append(f"{name}: Video_ID column nahi mila")
    else:
        missing_ids = int(df["Video_ID"].isna().sum())
        duplicate_ids = int(df["Video_ID"].duplicated().sum())
        if missing_ids or duplicate_ids:
            problems.append(
                f"{name}: missing Video_ID={missing_ids}, "
                f"duplicate Video_ID={duplicate_ids}"
            )

    labels = [col for col in df.columns if col != "Video_ID"]
    missing_labels = int(df[labels].isna().sum().sum())

    # Require binary labels with values zero or one.
    for col in labels:
        numeric = pd.to_numeric(df[col], errors="coerce")
        invalid = int((~numeric.isin([0, 1]) & df[col].notna()).sum())
        if invalid:
            problems.append(f"{name}: '{col}' mein {invalid} invalid values")
        df[col] = numeric

    if missing_labels:
        problems.append(f"{name}: {missing_labels} missing label values")

    cleaned[name] = df
    print(
        f"{name}: {original_rows} → {len(df)} rows | "
        f"empty removed={empty_rows} | exact duplicates removed={duplicate_rows}"
    )

# Check clip and original-video overlap across splits.
def source_video(video_id):
    match = re.fullmatch(r"(.+)_\d+_\d+", str(video_id))
    return match.group(1) if match else str(video_id)

splits = ["train", "val", "test"]
for i, first in enumerate(splits):
    for second in splits[i + 1:]:
        a = cleaned[first]["Video_ID"]
        b = cleaned[second]["Video_ID"]

        clip_overlap = len(set(a.dropna()) & set(b.dropna()))
        source_overlap = len(
            set(a.dropna().map(source_video))
            & set(b.dropna().map(source_video))
        )
        print(
            f"{first}/{second}: clip overlap={clip_overlap}, "
            f"source overlap={source_overlap}"
        )

        if clip_overlap or source_overlap:
            problems.append(f"{first}/{second}: split overlap mila")

# Compare dataset IDs with combined split IDs.
dataset_ids = set(cleaned["dataset"]["Video_ID"].dropna())
split_ids = set(
    pd.concat([cleaned[s]["Video_ID"] for s in splits]).dropna()
)
print(
    "Dataset se splits mein missing clips:",
    len(dataset_ids - split_ids)
)
print(
    "Splits mein extra clips:",
    len(split_ids - dataset_ids)
)

if dataset_ids != split_ids:
    problems.append("dataset.csv aur train/val/test ke Video_ID match nahi karte")

# Save cleaned files only when validation has no issues.
if problems:
    print("\nISSUES:")
    for problem in problems:
        print("-", problem)
    print("\nCleaned files abhi save nahi ki gayin.")
else:
    for name, df in cleaned.items():
        output = folder / f"{name}_clean.csv"
        df.to_csv(output, index=False)
        print("Saved:", output)

    print("\nBasic CSV cleaning aur checks complete.")

## Available Original Videos

In [ ]:
from pathlib import Path
import re
import pandas as pd


# Count completed videos and exclude temporary format files.
available = {
    p.stem
    for p in video_folder.iterdir()
    if p.is_file()
    and p.suffix.lower() in {".mp4", ".mkv", ".webm"}
    and p.stat().st_size > 0
    and not re.search(r"\.f\d+$", p.stem)
}

print("Available original video files:", len(available))
print("Incomplete .part files:", len(list(video_folder.glob("*.part"))))

for name in ["dataset", "train", "val", "test"]:
    df = pd.read_csv(project / "dataset" / "csvs" / f"{name}_clean.csv")
    sources = df["Video_ID"].astype(str).map(
        lambda value: value.rsplit("_", 2)[0]
    )
    covered = sources.isin(available).sum()
    print(f"{name}: available clips = {covered}/{len(df)}")

## FFmpeg Tool Check

In [ ]:
import shutil
import subprocess

for tool in ["ffmpeg", "ffprobe"]:
    path = shutil.which(tool)
    print(f"{tool}: {path or 'NOT FOUND'}")

    if path:
        result = subprocess.run(
            [path, "-version"],
            capture_output=True,
            text=True
        )
        print(result.stdout.splitlines()[0])
        

## Source Metadata and Timestamp Validation

In [ ]:
from pathlib import Path
import json
import re
import shutil
import subprocess
import pandas as pd

ffprobe = shutil.which("ffprobe")
if not ffprobe:
    raise RuntimeError("FFprobe must be available.")

df = pd.read_csv(project / "dataset/csvs/dataset_clean.csv")

# Find completed video files and exclude temporary format files.
video_files = {}
for p in video_folder.iterdir():
    if (
        p.is_file()
        and p.suffix.lower() in {".mp4", ".mkv", ".webm"}
        and p.stat().st_size > 0
        and not re.search(r"\.f\d+$", p.stem)
    ):
        video_files.setdefault(p.stem, []).append(p)

metadata = {}

# Check the duration and streams of each available video.
for number, (source_id, paths) in enumerate(video_files.items(), 1):
    candidates = []

    for path in paths:
        try:
            result = subprocess.run(
                [
                    ffprobe, "-v", "error",
                    "-show_entries", "format=duration:stream=codec_type",
                    "-of", "json", str(path)
                ],
                capture_output=True,
                text=True,
                timeout=30,
                check=True
            )

            info = json.loads(result.stdout)
            duration = float(info.get("format", {}).get("duration", 0))
            streams = {
                stream.get("codec_type")
                for stream in info.get("streams", [])
            }

            if duration > 0 and "video" in streams:
                candidates.append({
                    "path": str(path),
                    "duration": duration,
                    "audio": "audio" in streams
                })

        except (subprocess.SubprocessError, ValueError, TypeError):
            pass

    # Prefer a video with audio when multiple formats are available.
    metadata[source_id] = (
        max(candidates, key=lambda item: (item["audio"], item["duration"]))
        if candidates else None
    )

    if number % 25 == 0:
        print(f"Checked {number}/{len(video_files)} video IDs")

records = []

# Validate clip timestamps against the original video duration.
for video_id in df["Video_ID"].astype(str):
    record = {"Video_ID": video_id, "status": ""}

    try:
        source_id, start, end = video_id.rsplit("_", 2)
        start, end = float(start), float(end)
        record.update(Source_ID=source_id, start=start, end=end)

        if start < 0 or end <= start:
            record["status"] = "invalid_clip_times"

        elif source_id not in metadata:
            record["status"] = "missing_video"

        elif metadata[source_id] is None:
            record["status"] = "metadata_check_failed"

        else:
            meta = metadata[source_id]
            record.update(meta)

            if end > meta["duration"]:
                record["status"] = "clip_exceeds_video_duration"

            elif not meta["audio"]:
                record["status"] = "video_without_audio"

            else:
                record["status"] = "metadata_ok"

    except ValueError:
        record["status"] = "invalid_Video_ID"

    records.append(record)

# Save the validation report without modifying the original videos.
report = pd.DataFrame(records)
output = project / "dataset/csvs/video_check_report.csv"
report.to_csv(output, index=False)

print("\nClip check results:")
print(report["status"].value_counts().to_string())
print("\nSaved:", output)

## Inspect Invalid Intervals

In [ ]:
from pathlib import Path
import pandas as pd

report = pd.read_csv(project / "dataset/csvs/video_check_report.csv")

# Inspect clips whose end timestamps exceed the video duration.
issues = report[
    report["status"] == "clip_exceeds_video_duration"
].copy()

issues["extra_seconds"] = issues["end"] - issues["duration"]

print(issues[
    ["Video_ID", "start", "end", "duration", "extra_seconds"]
].to_string(index=False))

# Count clips that passed metadata checks in each original split.
valid_ids = set(
    report.loc[report["status"] == "metadata_ok", "Video_ID"]
)

for split in ["train", "val", "test"]:
    df = pd.read_csv(project / f"dataset/csvs/{split}_clean.csv")
    print(f"{split}: {df['Video_ID'].isin(valid_ids).sum()} metadata_ok clips")

## Clip Extraction and Verification

In [ ]:
from pathlib import Path
import json
import shutil
import subprocess
import pandas as pd

output_folder = project / "dataset/clips"
output_folder.mkdir(parents=True, exist_ok=True)

ffmpeg = shutil.which("ffmpeg")
ffprobe = shutil.which("ffprobe")
if not ffmpeg or not ffprobe:
    raise RuntimeError("FFmpeg and FFprobe must be available.")

report = pd.read_csv(project / "dataset/csvs/video_check_report.csv")

# Process all clips that passed metadata validation.
clips = report[report["status"] == "metadata_ok"]
results = []

for row in clips.itertuples(index=False):
    output = output_folder / f"{row.Video_ID}.mp4"
    temporary = output_folder / f"{row.Video_ID}.partial.mp4"
    expected_duration = float(row.end - row.start)

    try:
        # Create an accurately trimmed clip using one encoding thread.
        if not output.exists():
            command = [
                ffmpeg, "-hide_banner", "-loglevel", "error",
                "-nostdin", "-y", "-xerror",
                "-ss", str(row.start),
                "-i", str(row.path),
                "-t", str(expected_duration),
                "-map", "0:v:0",
                "-map", "0:a:0",
                "-c:v", "libx264",
                "-preset", "veryfast",
                "-crf", "23",
                "-threads", "1",
                "-c:a", "aac",
                "-b:a", "128k",
                "-movflags", "+faststart",
                str(temporary)
            ]
            subprocess.run(
                command, capture_output=True, text=True,
                check=True, timeout=1800
            )

        candidate = output if output.exists() else temporary

        # Verify the output duration and required audio/video streams.
        probe = subprocess.run(
            [
                ffprobe, "-v", "error",
                "-show_entries", "format=duration:stream=codec_type",
                "-of", "json", str(candidate)
            ],
            capture_output=True, text=True,
            check=True, timeout=30
        )
        info = json.loads(probe.stdout)
        actual_duration = float(info["format"]["duration"])
        streams = {s["codec_type"] for s in info["streams"]}

        if not {"video", "audio"}.issubset(streams):
            raise ValueError("Output is missing video or audio.")

        if abs(actual_duration - expected_duration) > 0.3:
            raise ValueError(
                f"Duration mismatch: expected {expected_duration:.2f}s, "
                f"received {actual_duration:.2f}s"
            )

        # Rename the temporary file only after validation succeeds.
        if candidate == temporary:
            temporary.replace(output)

        results.append((row.Video_ID, "PASS"))
        print(f"PASS: {row.Video_ID} | {actual_duration:.2f}s")

    except Exception as error:
        details = (
            error.stderr
            if isinstance(error, subprocess.CalledProcessError)
            else str(error)
        )
        results.append((row.Video_ID, "FAIL"))
        print(f"FAIL: {row.Video_ID} | {details}")

print("\nPassed:", sum(status == "PASS" for _, status in results))
print("Failed:", sum(status == "FAIL" for _, status in results))
print("Clip folder:", output_folder)

# Persist clip verification results for later inspection.
pd.DataFrame(results, columns=["Video_ID", "status"]).to_csv(
    csv_folder / "clip_verification_report.csv", index=False
)


## Build Ready Annotation Subsets

In [ ]:
from pathlib import Path
from itertools import combinations
import pandas as pd

# Set project paths.
csv_folder = project / "dataset" / "csvs"
clip_folder = project / "dataset" / "clips"

# Load the source validation report.
report = pd.read_csv(csv_folder / "video_check_report.csv")
valid_ids = set(
    report.loc[report["status"] == "metadata_ok", "Video_ID"]
    .astype(str)
    .str.strip()
)

# Confirm every expected clip exists and is non-empty.
missing_clips = [
    video_id
    for video_id in sorted(valid_ids)
    if not (clip_folder / f"{video_id}.mp4").is_file()
    or (clip_folder / f"{video_id}.mp4").stat().st_size == 0
]

if missing_clips:
    raise ValueError(
        f"{len(missing_clips)} expected clips are missing or empty: "
        f"{missing_clips[:10]}"
    )

# Require current verification evidence before preparing ready subsets.
verification = pd.read_csv(csv_folder / "clip_verification_report.csv")
if verification["Video_ID"].duplicated().any():
    raise ValueError("Duplicate clip IDs in verification report.")
passed_ids = set(verification.loc[verification["status"] == "PASS", "Video_ID"])
if not valid_ids.issubset(passed_ids):
    raise ValueError("Some metadata-valid clips did not pass extraction verification.")

# Keep the original train, validation, and test assignments.
ready = {}

for split in ("train", "val", "test"):
    df = pd.read_csv(csv_folder / f"{split}_clean.csv")
    df["Video_ID"] = df["Video_ID"].astype(str).str.strip()

    if df["Video_ID"].duplicated().any():
        raise ValueError(f"Duplicate Video_ID found in {split}.")

    ready[split] = df.loc[df["Video_ID"].isin(valid_ids)].copy()

# Check clip and source-video separation before saving.
for first, second in combinations(ready, 2):
    first_ids = set(ready[first]["Video_ID"])
    second_ids = set(ready[second]["Video_ID"])

    first_sources = {
        video_id.rsplit("_", 2)[0] for video_id in first_ids
    }
    second_sources = {
        video_id.rsplit("_", 2)[0] for video_id in second_ids
    }

    clip_overlap = first_ids & second_ids
    source_overlap = first_sources & second_sources

    if clip_overlap or source_overlap:
        raise ValueError(f"Overlap detected between {first} and {second}.")

    print(f"{first}/{second}: clip overlap=0, source overlap=0")

# Confirm the splits cover all expected verified clips.
combined = pd.concat(ready.values(), ignore_index=True)

if set(combined["Video_ID"]) != valid_ids:
    raise ValueError("Split coverage does not match the valid clip IDs.")

# Save separate CSVs without overwriting the cleaned originals.
for split, df in ready.items():
    output = csv_folder / f"{split}_ready.csv"
    df.to_csv(output, index=False)
    print(f"{split}: {len(df)} clips | Saved: {output.name}")

combined.to_csv(csv_folder / "dataset_ready.csv", index=False)
print(f"\nTotal ready clips: {len(combined)}")
print("Ready CSV files saved successfully.")